# Imports

In [20]:
import pandas as pd
import numpy as np
import re

## Load the input table (microbial related citations until 2025)

In [21]:
df = pd.read_csv("../docs/extended/extended_data_table_3.tsv", sep="\t")
print(f"Total number of rows: {len(df)}")

Total number of rows: 3067


# Selecting a fixed random percentage from the microbial related citations

## Extracting a reproducible random 1% sample and saving it

In [22]:
SEED = 42  # a fixed integer to garanty reproducibility
np.random.seed(SEED)

sample_fraction = 0.01  # 1% of the data
sample_size = int(len(df) * sample_fraction)

sampled_df = df.sample(
    n=sample_size,
    random_state=SEED
)

print(f"Selected {len(sampled_df)} rows ({sample_fraction*100:.0f}%)")

Selected 30 rows (1%)


## Saving the extracted citations

In [25]:
sampled_df = sampled_df.sort_index()
sampled_df.head()
sampled_df.to_csv( "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations.tsv", sep="\t", index=False )

# Calculating stats between classifier (automatic) and manual validation

## Files paths and category mapping

In [26]:
VALIDATION_PATH = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation.tsv"
OUTPUT_ROW_LEVEL = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_row_level_keyword_confusion.tsv"
OUTPUT_SUMMARY = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_summary_keyword_confusion.tsv"

CATEGORIES = {
    "Targeted organisms": "Manual Targeted organisms",
    "Technical target": "Manual Technical target",
    "Methods": "Manual Methods",
    "Topics": "Manual Topics"
}

## Identifying the universal keywords for each category (mainly for calculationg True Negative)

In [27]:
LABEL_UNIVERSE_RAW = {
    "Targeted organisms": [
        "Bacteria", "virus", "archaea", "eukaryot", "microbiome", "pathogen"
    ],
    "Technical target": [
        "Isolate",
        "communits (taxonomy) profiling",
        "functional analysis",
        "interactome",
        "amr",
        "MAGs",
        "gene identification/biomarker",
        "SNP",
        "(M)LST",
        "Annotation",
        "variant",
        "comparative analysis"
    ],
    "Methods": [
        "metabarcoding",
        "(meta)genomics",
        "metagenomics",
        "(meta)transcriptomics",
        "metatranscriptomics",
        "(meta)proteomics",
        "metabolomics",
        "imaging"
    ],
    "Topics": [
        "Ecosystem Dnamics & Biodiversity",
        "Health&Disease"
    ]
}

## Normalising keywords (canonicalisation)

In [28]:
CANONICAL = {
    # Targeted organisms variants
    "eukaryot": "eukaryote",
    "eukaryote": "eukaryote",
    "eukaryotes": "eukaryote",

    # Technical target variants
    "communits (taxonomy) profiling": "community (taxonomy) profiling",
    "community (taxonomy) profiling": "community (taxonomy) profiling",
    "communities (taxonomy) profiling": "community (taxonomy) profiling",

    "(m)lst": "(m)lst",
    "(mlst)": "(m)lst",
    "mlst": "(m)lst",
    "mst": "(m)lst",

    "mags": "mags",
    "mag": "mags",

    "snp": "snp",
    "snps": "snp",

    # Topics variants
    "ecosystem dnamics & biodiversity": "ecosystem dynamics & biodiversity",
    "ecosystem dynamics & biodiversity": "ecosystem dynamics & biodiversity",

    "health&disease": "health&disease",
    "health & disease": "health&disease",
    "health &disease": "health&disease",
    "health and disease": "health&disease",
    "health & disease ": "health&disease",
}
def normalize_label(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return CANONICAL.get(s, s)

def cell_to_set(cell) -> set:
    """
    Convert a comma-separated string cell to a normalized set of keywords.
    - Empty/NaN -> empty set
    - lowercases, trims, collapses spaces
    - canonicalizes known variants/typos
    """
    if pd.isna(cell):
        return set()
    s = str(cell).strip()
    if not s:
        return set()

    parts = [p.strip() for p in s.split(",")]
    parts = [p for p in parts if p.strip()]
    return set(normalize_label(p) for p in parts)

def safe_div(num, den):
    return float(num) / float(den) if den else np.nan

## loading manual validation table and checking its columns

In [29]:
df = pd.read_csv(VALIDATION_PATH, sep="\t")
print("Validation sample size:", len(df))

missing_cols = [c for c in list(CATEGORIES.keys()) + list(CATEGORIES.values()) if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in validation file: {missing_cols}")

df.head()

Validation sample size: 30


,Unnamed: 0,year,title,abstract,Targeted organisms,Technical target,Methods,Topics,Manual Targeted organisms,Manual Technical target,Manual Methods,Manual Topics
0,106,2025,Clinical assessment and transcriptome analysis...,A glycoprotein-G-deleted live-attenuated vacci...,"Virus, Pathogen",Functional analysis,(Meta)transcriptomics,"Ecosystem Dynamics & Biodiversity, Health & Di...","Virus, Pathogen","Functional analysis, Gene identification / Bio...",(Meta)transcriptomics,Health & Disease
1,207,2024,Gene expression reprogramming of Pseudomonas a...,Abstract Different bacteria change their life ...,Bacteria,Functional analysis,(Meta)transcriptomics,Ecosystem Dynamics & Biodiversity,Bacteria,"Functional analysis, Gene identification / Bio...",(Meta)transcriptomics,Ecosystem Dynamics & Biodiversity
2,270,2023,Systematic SARS-CoV-2 S Gene Sequencing in Was...,"As the COVID-19 pandemic reached its peak, man...","Virus, Pathogen","Community (taxonomy) profiling, Variant",(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di...","Virus, Pathogen","SNP, Variant, Comparative analysis",(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di..."
3,490,2023,Comparative Microbiome Analysis of Three Epide...,(1) Background: Amplicon-based 16S rRNA profil...,"Bacteria, Microbiome","Community (taxonomy) profiling, Comparative an...",Metabarcoding,"Ecosystem Dynamics & Biodiversity, Health & Di...","Bacteria, Microbiome","Community (taxonomy) profiling, Comparative an...",Metabarcoding,"Ecosystem Dynamics & Biodiversity, Health & Di..."
4,1096,2025,Genomic Differences Between Two Fusarium oxysp...,The host specificity of Fusarium oxysporum (Fo...,Pathogen,NaN,(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di...","Pathogen, Eukaryote","Gene identification / Biomarker, Annotation, V...",(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di..."


## Building normalised universes (Per category)

In [30]:
label_universe = {}
for cat, labels in LABEL_UNIVERSE_RAW.items():
    label_universe[cat] = set(normalize_label(x) for x in labels)

# Quick sanity check: every category in CATEGORIES should have a universe
for cat in CATEGORIES.keys():
    if cat not in label_universe:
        raise ValueError(f"Missing label universe definition for category: {cat}")
    print(f"Universe size for '{cat}': {len(label_universe[cat])}")

Universe size for 'Targeted organisms': 6
Universe size for 'Technical target': 12
Universe size for 'Methods': 8
Universe size for 'Topics': 2


## Compute per-publication (row-level) TP/FP/FN/TN + audit labels

In [31]:
row_level_records = []

# Keep some ID columns if present
possible_id_cols = ["year", "title", "doi", "DOI", "PublicationID", "pub_id", "pmid"]
id_cols = [c for c in possible_id_cols if c in df.columns]

for idx, row in df.iterrows():
    out = {"row_index": idx}
    for c in id_cols:
        out[c] = row.get(c)

    for auto_col, manual_col in CATEGORIES.items():
        A = cell_to_set(row[auto_col])       # automatic keywords
        M = cell_to_set(row[manual_col])     # manual keywords
        U = label_universe[auto_col]         # fixed universe keywords

        tp_set = A & M
        fp_set = A - M
        fn_set = M - A
        tn_set = U - (A | M)

        out[f"{auto_col} TP"] = len(tp_set)
        out[f"{auto_col} FP"] = len(fp_set)
        out[f"{auto_col} FN"] = len(fn_set)
        out[f"{auto_col} TN"] = len(tn_set)

        # Store labels for inspection / audit trail
        out[f"{auto_col} TP_labels"] = ", ".join(sorted(tp_set))
        out[f"{auto_col} FP_labels"] = ", ".join(sorted(fp_set))
        out[f"{auto_col} FN_labels"] = ", ".join(sorted(fn_set))

    row_level_records.append(out)

row_level_df = pd.DataFrame(row_level_records)
print("Row-level table created:", row_level_df.shape)

row_level_df.to_csv(OUTPUT_ROW_LEVEL, sep="\t", index=False)
print("Saved:", OUTPUT_ROW_LEVEL)

row_level_df.head()

Row-level table created: (30, 31)
Saved: ../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_row_level_keyword_confusion.tsv


,row_index,year,title,Targeted organisms TP,Targeted organisms FP,Targeted organisms FN,Targeted organisms TN,Targeted organisms TP_labels,Targeted organisms FP_labels,Targeted organisms FN_labels,...,Methods TP_labels,Methods FP_labels,Methods FN_labels,Topics TP,Topics FP,Topics FN,Topics TN,Topics TP_labels,Topics FP_labels,Topics FN_labels
0,0,2025,Clinical assessment and transcriptome analysis...,2,0,0,4,"pathogen, virus",,,...,(meta)transcriptomics,,,1,1,0,0,health&disease,ecosystem dynamics & biodiversity,
1,1,2024,Gene expression reprogramming of Pseudomonas a...,1,0,0,5,bacteria,,,...,(meta)transcriptomics,,,1,0,0,1,ecosystem dynamics & biodiversity,,
2,2,2023,Systematic SARS-CoV-2 S Gene Sequencing in Was...,2,0,0,4,"pathogen, virus",,,...,(meta)genomics,,,2,0,0,0,"ecosystem dynamics & biodiversity, health&disease",,
3,3,2023,Comparative Microbiome Analysis of Three Epide...,2,0,0,4,"bacteria, microbiome",,,...,metabarcoding,,,2,0,0,0,"ecosystem dynamics & biodiversity, health&disease",,
4,4,2025,Genomic Differences Between Two Fusarium oxysp...,1,0,1,4,pathogen,,eukaryote,...,(meta)genomics,,,2,0,0,0,"ecosystem dynamics & biodiversity, health&disease",,


## Aggregate to category-level TP/FP/FN/TN and compute Precision/Recall/F1

Definitions (keyword-level, per publication, per category)

Let A be the set of automatic keywords in a cell, M the set of manual keywords, and U the fixed keyword universe for that category.

TP = |A ∩ M| (keywords correctly assigned)

FP = |A − M| (keywords assigned automatically but not confirmed manually)

FN = |M − A| (keywords missing from the automatic assignment)

TN = |U − (A ∪ M)| (keywords correctly not assigned by either)

This allows partial agreement: if automatic assigns multiple labels and only some match manual, those contribute to both TP and FP.

In [32]:
summary_records = []

for auto_col, manual_col in CATEGORIES.items():
    TP = int(row_level_df[f"{auto_col} TP"].sum())
    FP = int(row_level_df[f"{auto_col} FP"].sum())
    FN = int(row_level_df[f"{auto_col} FN"].sum())
    TN = int(row_level_df[f"{auto_col} TN"].sum())

    precision = safe_div(TP, TP + FP)
    recall = safe_div(TP, TP + FN)
    f1 = safe_div(2 * precision * recall, precision + recall)

    summary_records.append({
        "Category": auto_col,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Precision_%": None if np.isnan(precision) else round(precision * 100, 1),
        "Recall_%": None if np.isnan(recall) else round(recall * 100, 1),
        "F1_%": None if np.isnan(f1) else round(f1 * 100, 1),
        "Universe_size": len(label_universe[auto_col]),
    })

summary_df = pd.DataFrame(summary_records)

print("\nCategory-level summary:")
print(summary_df.to_string(index=False))

summary_df.to_csv(OUTPUT_SUMMARY, sep="\t", index=False)
print("Saved:", OUTPUT_SUMMARY)



Category-level summary:
          Category  TP  FP  FN  TN  Precision   Recall       F1  Precision_%  Recall_%  F1_%  Universe_size
Targeted organisms  34  11  20 115   0.755556 0.629630 0.686869         75.6      63.0  68.7              6
  Technical target  22  14  72 268   0.611111 0.234043 0.338462         61.1      23.4  33.8             12
           Methods  27   2   8 203   0.931034 0.771429 0.843750         93.1      77.1  84.4              8
            Topics  35   5   4  16   0.875000 0.897436 0.886076         87.5      89.7  88.6              2
Saved: ../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_summary_keyword_confusion.tsv


## Compute micro and macro averages

Macro vs Micro averaging

Macro-average: compute the metric per category, then average across categories.
Each category contributes equally.

Micro-average: sum TP/FP/FN across categories first, then compute the metric once.
Categories with more keyword decisions contribute more.

We report both to show overall performance (micro) and balance across categories (macro).

In [33]:
macro_precision = summary_df["Precision"].mean()
macro_recall = summary_df["Recall"].mean()
macro_f1 = summary_df["F1"].mean()

total_TP = int(summary_df["TP"].sum())
total_FP = int(summary_df["FP"].sum())
total_FN = int(summary_df["FN"].sum())

micro_precision = safe_div(total_TP, total_TP + total_FP)
micro_recall = safe_div(total_TP, total_TP + total_FN)
micro_f1 = safe_div(2 * micro_precision * micro_recall, micro_precision + micro_recall)

print("\nAverages:")
print(f"Macro Precision: {macro_precision:.3f} ({macro_precision*100:.1f}%)")
print(f"Macro Recall:    {macro_recall:.3f} ({macro_recall*100:.1f}%)")
print(f"Macro F1:        {macro_f1:.3f} ({macro_f1*100:.1f}%)")
print(f"Micro Precision: {micro_precision:.3f} ({micro_precision*100:.1f}%)")
print(f"Micro Recall:    {micro_recall:.3f} ({micro_recall*100:.1f}%)")
print(f"Micro F1:        {micro_f1:.3f} ({micro_f1*100:.1f}%)")

# -----------------------------
# Summarised stats for the paper
# -----------------------------
n_publications = len(df)

f1_min = float(summary_df["F1_%"].min())
f1_max = float(summary_df["F1_%"].max())
p_min = float(summary_df["Precision_%"].min())
p_max = float(summary_df["Precision_%"].max())
r_min = float(summary_df["Recall_%"].min())
r_max = float(summary_df["Recall_%"].max())



Averages:
Macro Precision: 0.793 (79.3%)
Macro Recall:    0.633 (63.3%)
Macro F1:        0.689 (68.9%)
Micro Precision: 0.787 (78.7%)
Micro Recall:    0.532 (53.2%)
Micro F1:        0.634 (63.4%)


# Validating microbial related chosen citations by classifier

## Reload microbial-related table and load all-citations table

In [35]:
# Reload already-loaded microbial-related table
df_micro = pd.read_csv("../docs/extended/extended_data_table_3.tsv", sep="\t")

# Load all citations table (Extended Data Table 2)
df_all = pd.read_csv("../docs/extended/extended_data_table_2.tsv", sep="\t")

print("Microbial-related rows (ED Table 3):", len(df_micro))
print("All citations rows (ED Table 2):", len(df_all))

Microbial-related rows (ED Table 3): 3067
All citations rows (ED Table 2): 10991


## Normalize titles and build key sets, to avoid mismatches due to spacing/case

In [36]:
def normalize_title(s) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)  # collapse whitespace
    return s

# Add normalized title keys (do not overwrite if already exists)
if "_title_key" not in df_micro.columns:
    df_micro["_title_key"] = df_micro["title"].apply(normalize_title)

if "_title_key" not in df_all.columns:
    df_all["_title_key"] = df_all["title"].apply(normalize_title)

micro_titles = set(df_micro["_title_key"].dropna().unique())
print("Unique microbial titles:", len(micro_titles))


Unique microbial titles: 3063


## Build the “known non-microbial” pool, all-citation rows whose titles are NOT in the microbial table

In [37]:
df_non_micro = df_all[~df_all["_title_key"].isin(micro_titles)].copy()

print("Non-microbial pool size (ED2 minus ED3):", len(df_non_micro))

# Optional sanity check:
overlap = df_all[df_all["_title_key"].isin(micro_titles)]
print("Overlap (should match ED3 titles present in ED2):", len(overlap))

Non-microbial pool size (ED2 minus ED3): 7923
Overlap (should match ED3 titles present in ED2): 3068


## Sample 5% + 5% (reproducible) and label yes/no

In [38]:
SEED = 42
sample_fraction_binary = 0.05  # 5%

micro_n = int(len(df_micro) * sample_fraction_binary)
non_micro_n = int(len(df_non_micro) * sample_fraction_binary)

print("Sampling sizes:")
print("  Microbial sample:", micro_n)
print("  Non-microbial sample:", non_micro_n)

micro_sample = df_micro.sample(n=micro_n, random_state=SEED).copy()
non_micro_sample = df_non_micro.sample(n=non_micro_n, random_state=SEED).copy()

micro_sample["microbial_related_auto"] = "yes"
non_micro_sample["microbial_related_auto"] = "no"

binary_validation_df = pd.concat([micro_sample, non_micro_sample], ignore_index=True)

# Shuffle for blind manual review (still reproducible)
binary_validation_df = binary_validation_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Final binary validation dataset size:", len(binary_validation_df))
print(binary_validation_df["microbial_related_auto"].value_counts())


Sampling sizes:
  Microbial sample: 153
  Non-microbial sample: 396
Final binary validation dataset size: 549
microbial_related_auto
no     396
yes    153
Name: count, dtype: int64


## Save the template to be used for manual annotation

In [39]:
OUTPUT_BINARY_SAMPLE = "../docs/extended/extended_data_table_2_3_binary_validation_5p_5p_auto.tsv"

binary_validation_df["microbial_related_manual"] = ""  # will be manually filled with yes/no

binary_validation_df.to_csv(OUTPUT_BINARY_SAMPLE, sep="\t", index=False)
print("Saved manual annotation template:", OUTPUT_BINARY_SAMPLE)

binary_validation_df.head()


Saved manual annotation template: ../docs/extended/extended_data_table_2_3_binary_validation_5p_5p_auto.tsv


,Unnamed: 0,year,title,abstract,Targeted organisms,Technical target,Methods,Topics,_title_key,microbial_related_auto,microbial_related_manual
0,5243,2017,In silico promoter analysis reveals rice Wx pr...,NaN,NaN,NaN,NaN,NaN,in silico promoter analysis reveals rice wx pr...,no,
1,2230,2019,Modulation of peptidoglycan synthesis by recyc...,The bacterial cell wall is made of peptidoglyc...,Bacteria,MAGs,NaN,"Ecosystem Dynamics & Biodiversity, Health & Di...",modulation of peptidoglycan synthesis by recyc...,yes,
2,6301,2023,Transcriptomic and targeted immune transcript ...,NaN,NaN,NaN,NaN,NaN,transcriptomic and targeted immune transcript ...,no,
3,178,2024,Unraveling tuberculosis patient cluster transm...,Background Tuberculosis remains a global healt...,Bacteria,"Isolate, AMR, Variant",(Meta)genomics,Health & Disease,unraveling tuberculosis patient cluster transm...,yes,
4,4138,2012,Golden Trail: Retrieving the Data History that...,NaN,NaN,NaN,NaN,NaN,golden trail: retrieving the data history that...,no,


## Compute TP/FP/FN/TN + Precision/Recall/F1/Accuracy

In [40]:
MANUAL_BINARY_PATH = "../docs/extended/extended_data_table_2_3_binary_validation_5p_5p_auto_manual.tsv"
df_bin = pd.read_csv(MANUAL_BINARY_PATH, sep="\t")

auto = df_bin["microbial_related_auto"].astype(str).str.strip().str.lower()
manual = df_bin["microbial_related_manual"].astype(str).str.strip().str.lower()

valid = auto.isin(["yes", "no"]) & manual.isin(["yes", "no"])
df_eval = df_bin[valid].copy()

auto = auto[valid]
manual = manual[valid]

TP = int(((auto == "yes") & (manual == "yes")).sum())
FP = int(((auto == "yes") & (manual == "no")).sum())
FN = int(((auto == "no") & (manual == "yes")).sum())
TN = int(((auto == "no") & (manual == "no")).sum())

precision = safe_div(TP, TP + FP)
recall = safe_div(TP, TP + FN)
f1 = safe_div(2 * precision * recall, precision + recall)
accuracy = safe_div(TP + TN, TP + TN + FP + FN)

print("Binary validation (microbiology-related yes/no):")
print("TP:", TP, "FP:", FP, "FN:", FN, "TN:", TN)
print(f"Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"F1:        {f1:.3f} ({f1*100:.1f}%)")
print(f"Accuracy:  {accuracy:.3f} ({accuracy*100:.1f}%)")
print("N evaluated:", len(df_eval))


Binary validation (microbiology-related yes/no):
TP: 134 FP: 19 FN: 38 TN: 358
Precision: 0.876 (87.6%)
Recall:    0.779 (77.9%)
F1:        0.825 (82.5%)
Accuracy:  0.896 (89.6%)
N evaluated: 549
